In [ ]:
# Cell 1 - Setup & Installs
!pip install transformers==4.40.0 mlflow==2.12.2 scikit-learn==1.4.2 evaluate==0.4.1
from google.colab import drive
drive.mount('/content/drive')
print('Setup complete. Drive mounted.')


In [ ]:
# Cell 2 - Config
from dataclasses import dataclass

@dataclass
class TrainConfig:
    MODEL_NAME = "distilbert-base-uncased"
    MAX_SEQ_LENGTH = 256
    BATCH_SIZE = 32
    EPOCHS = 4
    LEARNING_RATE = 2e-5
    TRAIN_PATH = "/content/data/processed/train.csv"
    VAL_PATH = "/content/data/processed/val.csv"
    TEST_PATH = "/content/data/processed/test.csv"
    MODEL_SAVE_PATH = "/content/drive/MyDrive/multi-label-text-clf/models/distilbert_multilabel"
    MLFLOW_TRACKING_URI = "/content/drive/MyDrive/multi-label-text-clf/mlruns"


In [ ]:
# Cell 3 - Data Loading
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast

# Load datasets
train_df = pd.read_csv(TrainConfig.TRAIN_PATH)
val_df = pd.read_csv(TrainConfig.VAL_PATH)
test_df = pd.read_csv(TrainConfig.TEST_PATH)

# Detect label columns
label_cols = [col for col in train_df.columns if col not in ['id', 'abstract']]
print(f"Detected {len(label_cols)} label columns.")

tokenizer = DistilBertTokenizerFast.from_pretrained(TrainConfig.MODEL_NAME)

class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, label_cols):
        self.texts = df['abstract'].fillna('').tolist()
        self.labels = df[label_cols].values
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        labels = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(labels, dtype=torch.float)
        }

train_dataset = EmotionDataset(train_df, tokenizer, TrainConfig.MAX_SEQ_LENGTH, label_cols)
val_dataset = EmotionDataset(val_df, tokenizer, TrainConfig.MAX_SEQ_LENGTH, label_cols)
test_dataset = EmotionDataset(test_df, tokenizer, TrainConfig.MAX_SEQ_LENGTH, label_cols)


In [ ]:
# Cell 4 - Model
from transformers import DistilBertForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = DistilBertForSequenceClassification.from_pretrained(
    TrainConfig.MODEL_NAME,
    num_labels=len(label_cols),
    problem_type="multi_label_classification"
)
model = model.to(device)


In [ ]:
# Cell 5 - MLflow + Training loop
import mlflow
import torch.nn as nn
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, hamming_loss
import numpy as np
from tqdm.auto import tqdm
import os

train_loader = DataLoader(train_dataset, batch_size=TrainConfig.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=TrainConfig.BATCH_SIZE)

optimizer = AdamW(model.parameters(), lr=TrainConfig.LEARNING_RATE)
total_steps = len(train_loader) * TrainConfig.EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
criterion = nn.BCEWithLogitsLoss()

mlflow.set_tracking_uri(TrainConfig.MLFLOW_TRACKING_URI)
mlflow.set_experiment("multi-label-text-clf")

best_val_f1 = 0.0

with mlflow.start_run(run_name="distilbert-multilabel-v1"):
    # Log params
    mlflow.log_params({
        "model_name": TrainConfig.MODEL_NAME,
        "max_seq_len": TrainConfig.MAX_SEQ_LENGTH,
        "batch_size": TrainConfig.BATCH_SIZE,
        "epochs": TrainConfig.EPOCHS,
        "learning_rate": TrainConfig.LEARNING_RATE
    })
    
    for epoch in range(TrainConfig.EPOCHS):
        print(f"\nEpoch {epoch+1}/{TrainConfig.EPOCHS}")
        model.train()
        total_train_loss = 0
        
        for batch in tqdm(train_loader, desc="Training"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            total_train_loss += loss.item()
            
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
        avg_train_loss = total_train_loss / len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0
        val_preds, val_labels = [], []
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = model(input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits, labels)
                val_loss += loss.item()
                
                preds = torch.sigmoid(outputs.logits) > 0.5
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
                
        avg_val_loss = val_loss / len(val_loader)
        val_f1_micro = f1_score(val_labels, val_preds, average="micro")
        val_f1_macro = f1_score(val_labels, val_preds, average="macro")
        val_hamming = hamming_loss(val_labels, val_preds)
        
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val F1 Micro: {val_f1_micro:.4f}")
        
        mlflow.log_metrics({
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "val_f1_micro": val_f1_micro,
            "val_f1_macro": val_f1_macro,
            "val_hamming_loss": val_hamming
        }, step=epoch)
        
        if val_f1_micro > best_val_f1:
            best_val_f1 = val_f1_micro
            print(f"New best model found! Saving to {TrainConfig.MODEL_SAVE_PATH}...")
            os.makedirs(TrainConfig.MODEL_SAVE_PATH, exist_ok=True)
            model.save_pretrained(TrainConfig.MODEL_SAVE_PATH)
            tokenizer.save_pretrained(TrainConfig.MODEL_SAVE_PATH)


In [ ]:
# Cell 6 - Evaluation
from sklearn.metrics import classification_report

test_loader = DataLoader(test_dataset, batch_size=TrainConfig.BATCH_SIZE)
best_model = DistilBertForSequenceClassification.from_pretrained(TrainConfig.MODEL_SAVE_PATH).to(device)
best_model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = best_model(input_ids, attention_mask=attention_mask)
        preds = torch.sigmoid(outputs.logits) > 0.5
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

test_f1_micro = f1_score(test_labels, test_preds, average="micro")
test_f1_macro = f1_score(test_labels, test_preds, average="macro")
test_hamming = hamming_loss(test_labels, test_preds)

print(f"Test F1 Micro: {test_f1_micro:.4f}")
print(f"Test F1 Macro: {test_f1_macro:.4f}")
print(f"Test Hamming Loss: {test_hamming:.4f}")

report = classification_report(test_labels, test_preds, target_names=label_cols, zero_division=0)
print("\nClassification Report:\n", report)

with mlflow.start_run(run_name="distilbert-multilabel-test", nested=True):
    mlflow.log_metrics({
        "test_f1_micro": test_f1_micro,
        "test_f1_macro": test_f1_macro,
        "test_hamming_loss": test_hamming
    })


### Instructions for running this notebook
1. Go to the left sidebar in Colab -> Click on the Folder icon (Files).
2. Create a folder path `/content/data/processed/`.
3. Upload `train.csv`, `val.csv`, and `test.csv` (generated locally by `src/preprocess.py`) into this folder.
4. Ensure your Google Drive is mounted and you have a folder path `/content/drive/MyDrive/multi-label-text-clf/` for MLflow and model saving.
5. Run all cells sequentially.
